In [4]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn"])

0

In [8]:
import pandas as pd
import numpy as np
import json
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load the dataset produced in Phase 2
df = pd.read_csv('model_table.csv')
df['visit_date'] = pd.to_datetime(df['visit_date'])

# For predicting clinical/operational Risk, we use patient and visit metrics.
# We do NOT use billing outcomes (approved_amount, claim_status) to avoid data leakage,
# because those happen *after* the visit risk is assessed.
# ── Cell 2: Feature Selection & Encode Target ────────────────────────────────
from sklearn.preprocessing import LabelEncoder

# Encode the target variable
le = LabelEncoder()
y = pd.Series(le.fit_transform(df['risk_score']))

# Verify the mapping (e.g. 0=High, 1=Low, 2=Medium)
target_mapping = dict(zip(le.transform(le.classes_), le.classes_))
print("Target Mapping:", target_mapping)
target_names = [target_mapping[i] for i in range(len(target_mapping))]

# Features for predicting Visit Risk
features = [
    'age', 'length_of_stay_hours', 'visit_frequency', 'avg_los_per_patient',
    'days_since_registration', 'visit_month', 'visit_dayofweek', 'chronic_flag',
    'department_enc', 'visit_type_enc', 'gender_enc', 'city_enc'
]

X = df[['visit_date'] + features].copy()

# Impute any edge-case missing values
X.fillna(X.median(numeric_only=True), inplace=True)


# Sort by date to ensure we train on the past and test on the future (Leakage-safe)
X = X.sort_values('visit_date')
y = y.loc[X.index]

split_idx = int(len(X) * 0.8)
X_train_full = X.iloc[:split_idx]
X_test_full  = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

# Drop the date column before modeling
X_train = X_train_full.drop(columns=['visit_date'])
X_test  = X_test_full.drop(columns=['visit_date'])

print(f"Train size: {len(X_train)} | Date range: {X_train_full['visit_date'].min().date()} to {X_train_full['visit_date'].max().date()}")
print(f"Test size:  {len(X_test)} | Date range: {X_test_full['visit_date'].min().date()} to {X_test_full['visit_date'].max().date()}")


# 'High' risk is the minority class (~20%). We apply SMOTE to the training data only.
print("\nOriginal y_train class counts:\n", y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nResampled y_train class counts:\n", y_train_resampled.value_counts())


baseline_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
baseline_model.fit(X_train_resampled, y_train_resampled)
y_pred_base = baseline_model.predict(X_test)

print("=== Baseline Model (Logistic Regression) ===")
print(classification_report(y_test, y_pred_base, target_names=target_names))


# We use moderate depth to prevent overfitting
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled)
y_pred_rf = rf_model.predict(X_test)

print("=== Advanced Model (Random Forest) ===")
print(classification_report(y_test, y_pred_rf, target_names=target_names))


joblib.dump(rf_model, 'risk_model.pkl')

schema = {
    "model_name": "Visit Risk Classification",
    "features": list(X_train.columns),
    "target_mapping": {0: "High", 1: "Low", 2: "Medium"} # Adjust mapping to match your LabelEncoder
}

with open('risk_feature_schema.json', 'w') as f:
    json.dump(schema, f)

print("\n risk_model.pkl and risk_feature_schema.json saved.")


Target Mapping: {0: 'High', 1: 'Low', 2: 'Medium'}
Train size: 20000 | Date range: 2025-01-20 to 2025-11-08
Test size:  5000 | Date range: 2025-11-08 to 2026-01-20

Original y_train class counts:
 1    9990
2    5999
0    4011
Name: count, dtype: int64

Resampled y_train class counts:
 0    9990
2    9990
1    9990
Name: count, dtype: int64
=== Baseline Model (Logistic Regression) ===
              precision    recall  f1-score   support

        High       0.19      0.26      0.22      1023
         Low       0.49      0.52      0.50      2480
      Medium       0.31      0.21      0.25      1497

    accuracy                           0.37      5000
   macro avg       0.33      0.33      0.33      5000
weighted avg       0.38      0.37      0.37      5000

=== Advanced Model (Random Forest) ===
              precision    recall  f1-score   support

        High       0.21      0.10      0.14      1023
         Low       0.49      0.64      0.56      2480
      Medium       0.31      

### Why we are doing this? 

### Business purpose:
The goal of this model is to predict whether an incoming hospital visit represents a Low, Medium, or High operational and clinical risk (risk_score). This predictive capability allows hospital administrators to proactively allocate ICU beds, adjust staffing levels, and triage high-risk patients efficiently.

1. Feature Selection & Leakage Prevention
To ensure the model is usable at the time of admission, all financial data (billing amounts, claim outcomes, payment delays) was strictly excluded. The model relies entirely on pre-treatment and demographic indicators available at triage, such as: Patient Profile: Age, gender, chronic condition flags, and visit frequency. Operational Context: Department, visit type, day of the week, and patient tenure (days since registration).

2. Time-Based Validation
Standard random splitting scrambles temporal patterns. To simulate how the model will perform in the real world (training on the past to predict the future), a time-based 80/20 split was applied using the visit_date. The earliest 80% of visits were used for training, and the final 20% were held out as a strict future-state test set.

3. Class Imbalance Strategy
Exploratory Data Analysis (EDA) revealed that "High" risk visits constituted only ~20% of the dataset. Because failing to identify a high-risk patient carries severe clinical consequences, we applied SMOTE (Synthetic Minority Over-sampling Technique) exclusively to the training data. This generated synthetic "High Risk" examples, allowing the algorithms to learn minority patterns without bleeding synthetic data into the test set.

4. Model Training

    Baseline: A Logistic Regression model was trained with class_weight='balanced' to establish a linear baseline of feature importance and predictive power.

    Advanced: A Random Forest Classifier (with max depth constraints to prevent overfitting) was trained to capture non-linear interactions, such as how specific combinations of age, chronic conditions, and departments compound visit risk.



In [14]:
import json

# Consolidated schema for both models to meet the exact rubric deliverable
consolidated_schema = {
    "model_a_risk": {
        "model_name": "Visit Risk Classification",
        "features": [
            "age", "length_of_stay_hours", "visit_frequency", "avg_los_per_patient",
            "days_since_registration", "visit_month", "visit_dayofweek", "chronic_flag",
            "department_enc", "visit_type_enc", "gender_enc", "city_enc"
        ],
        "target_mapping": {"0": "High", "1": "Low", "2": "Medium"}
    },
    "model_b_claim": {
        "model_name": "Claim Outcome Classification",
        "features": [
            "billed_amount", "provider_rejection_rate", "high_billed_flag",
            "department_enc", "insurance_provider_enc", "visit_type_enc",
            "age", "length_of_stay_hours", "chronic_flag"
        ],
        "target_mapping": {"0": "Paid", "1": "Pending", "2": "Rejected"}
    }
}

# Save as a single formatted JSON file
with open('feature_schema.json', 'w') as f:
    json.dump(consolidated_schema, f, indent=4)

print(" feature_schema.json created successfully!")


 feature_schema.json created successfully!
